In [0]:
train_bronze = spark.read.option("header", True).option("inferSchema", True) \
    .csv("/Volumes/workspace/climate_challenge/raw_files/Train.csv")

test_bronze = spark.read.option("header", True).option("inferSchema", True) \
    .csv("/Volumes/workspace/climate_challenge/raw_files/Test.csv")

climate_bronze = spark.read.option("header", True).option("inferSchema", True) \
    .csv("/Volumes/workspace/climate_challenge/raw_files/climate_features.csv")

print(train_bronze.count(), test_bronze.count(), climate_bronze.count())
train_bronze.display()

In [0]:
train_bronze.write.mode("overwrite").saveAsTable("workspace.climate_challenge.bronze_train")
test_bronze.write.mode("overwrite").saveAsTable("workspace.climate_challenge.bronze_test")
climate_bronze.write.mode("overwrite").saveAsTable("workspace.climate_challenge.bronze_climate")

In [0]:
%sql
select * from climate_challenge.bronze_train
where age >= 50

In [0]:
from pyspark.sql import functions as F

bronze_train = spark.table("workspace.climate_challenge.bronze_train")
bronze_test = spark.table("workspace.climate_challenge.bronze_test")
bronze_climate = spark.table("workspace.climate_challenge.bronze_climate").drop("deathdate")

# join train with the richer climate features, on ID
silver_train = bronze_train.join(bronze_climate, on="ID", how="left") \
    .withColumn("deathdate", F.to_date("deathdate")) \
    .withColumn("death_month", F.month("deathdate")) \
    .withColumn("death_year", F.year("deathdate")) \
    .dropDuplicates(["ID"])

silver_test = bronze_test.join(bronze_climate, on="ID", how="left") \
    .withColumn("deathdate", F.to_date("deathdate")) \
    .withColumn("death_month", F.month("deathdate")) \
    .withColumn("death_year", F.year("deathdate")) \
    .dropDuplicates(["ID"])

print("Silver train count:", silver_train.count())
print("Silver test count:", silver_test.count())

# 1. Check nulls per column
print("\n--- Null counts in silver_train ---")
silver_train.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in silver_train.columns]).display()

# 2. Check class balance
print("\n--- Class balance (is_climate_sensitive) ---")
silver_train.groupBy("is_climate_sensitive").count().display()

In [0]:
silver_train.write.mode("overwrite").saveAsTable("workspace.climate_challenge.silver_train")
silver_test.write.mode("overwrite").saveAsTable("workspace.climate_challenge.silver_test")

print("Silver tables created successfully")

In [0]:
from pyspark.sql import functions as F

silver_train = spark.table("workspace.climate_challenge.silver_train")
silver_test = spark.table("workspace.climate_challenge.silver_test")

# check unique values in categorical columns first, so we encode them correctly
silver_train.select("zone").distinct().display()
silver_train.select("gender").distinct().display()

In [0]:
from pyspark.sql import functions as F

silver_train = spark.table("workspace.climate_challenge.silver_train")
silver_test = spark.table("workspace.climate_challenge.silver_test")

def build_gold(df):
    return df.withColumn("zone_rural", F.when(F.col("zone") == "Rural", 1).otherwise(0)) \
              .withColumn("gender_male", F.when(F.col("gender") == "Male", 1).otherwise(0)) \
              .drop("zone", "gender", "deathdate", "location")

gold_train = build_gold(silver_train)
gold_test = build_gold(silver_test)

print("Gold train columns:", gold_train.columns)
gold_train.display()

In [0]:
# save as a regular Delta table
gold_train.write.mode("overwrite").saveAsTable("workspace.climate_challenge.gold_train")
gold_test.write.mode("overwrite").saveAsTable("workspace.climate_challenge.gold_test")

print("Gold tables created successfully")

In [0]:
%pip install databricks-feature-engineering

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# feature table ecluding the target column
gold_train_features = gold_train.drop("is_climate_sensitive")

fe.create_table(
    name="workspace.climate_challenge.gold_features",
    primary_keys=["ID"],
    df=gold_train_features,
    description="Model-ready features for climate-sensitive mortality prediction"
)

print("Feature table registered successfully")

In [0]:
import mlflow

mlflow.set_experiment("/Users/nvethiappan@gmail.com/climate_mortality_experiment")
print("MLflow experiment set")

In [0]:
gold_train_pd = spark.table("workspace.climate_challenge.gold_train").toPandas()
gold_test_pd = spark.table("workspace.climate_challenge.gold_test").toPandas()

print(gold_train_pd.shape, gold_test_pd.shape)
gold_train_pd.head()

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score
import mlflow
import mlflow.sklearn

drop_cols = ['ID', 'is_climate_sensitive']
feature_cols = [c for c in gold_train_pd.columns if c not in drop_cols]

X = gold_train_pd[feature_cols]
y = gold_train_pd['is_climate_sensitive']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

with mlflow.start_run(run_name="baseline_logreg_scaled"):
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train_scaled, y_train)

    preds = model.predict(X_val_scaled)
    probs = model.predict_proba(X_val_scaled)[:, 1]

    f1 = f1_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)

    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("scaled", True)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", auc)
    mlflow.sklearn.log_model(model, "model", input_example=X_train_scaled[:5])

    print(f"F1: {f1:.4f}")
    print(f"ROC-AUC: {auc:.4f}")

In [0]:
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
gold_train_pd = spark.table("workspace.climate_challenge.gold_train").toPandas()
gold_test_pd = spark.table("workspace.climate_challenge.gold_test").toPandas()

from sklearn.model_selection import train_test_split

drop_cols = ['ID', 'is_climate_sensitive']
feature_cols = [c for c in gold_train_pd.columns if c not in drop_cols]

X = gold_train_pd[feature_cols]
y = gold_train_pd['is_climate_sensitive']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Reloaded and split successfully:", X_train.shape, X_val.shape)

In [0]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score
import mlflow
import mlflow.xgboost

with mlflow.start_run(run_name="xgboost_baseline"):
    # scale_pos_weight helps with class imbalance for XGBoost specifically
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model_xgb = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    )
    model_xgb.fit(X_train, y_train)

    preds = model_xgb.predict(X_val)
    probs = model_xgb.predict_proba(X_val)[:, 1]

    f1 = f1_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)

    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", auc)
    mlflow.xgboost.log_model(model_xgb, "model", input_example=X_train[:5])

    print(f"F1: {f1:.4f}")
    print(f"ROC-AUC: {auc:.4f}")

In [0]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score
import mlflow

with mlflow.start_run(run_name="xgboost_tuned"):
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model_xgb2 = XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    )
    model_xgb2.fit(X_train, y_train)

    preds = model_xgb2.predict(X_val)
    probs = model_xgb2.predict_proba(X_val)[:, 1]

    f1 = f1_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)

    mlflow.log_param("model_type", "XGBoost_tuned")
    mlflow.log_param("n_estimators", 400)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", auc)
    mlflow.xgboost.log_model(model_xgb2, "model", input_example=X_train[:5])

    print(f"F1: {f1:.4f}")
    print(f"ROC-AUC: {auc:.4f}")

In [0]:
import pandas as pd

X_test = gold_test_pd[feature_cols]

test_preds = model_xgb2.predict(X_test)
test_probs = model_xgb2.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'ID': gold_test_pd['ID'],
    'TargetF1': test_preds,
    'TargetRAUC': test_probs
})

submission.to_csv('/Volumes/workspace/climate_challenge/raw_files/submission_xgb_tuned.csv', index=False)
print(submission.head())
print("Shape:", submission.shape)

In [0]:
%pip install optuna

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split

gold_train_pd = spark.table("workspace.climate_challenge.gold_train").toPandas()
gold_test_pd = spark.table("workspace.climate_challenge.gold_test").toPandas()

drop_cols = ['ID', 'is_climate_sensitive']
feature_cols = [c for c in gold_train_pd.columns if c not in drop_cols]

X = gold_train_pd[feature_cols]
y = gold_train_pd['is_climate_sensitive']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [0]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score
import mlflow

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'logloss',
        'random_state': 42
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:, 1]

    f1 = f1_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)
    weighted_score = 0.6 * f1 + 0.4 * auc  # matches Zindi's actual scoring

    return weighted_score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)  # 30 trials is a reasonable, fast search

print("Best params:", study.best_params)
print("Best weighted score:", study.best_value)

In [0]:
X_test = gold_test_pd[feature_cols]

test_preds = final_model.predict(X_test)
test_probs = final_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'ID': gold_test_pd['ID'],
    'TargetF1': test_preds,
    'TargetRAUC': test_probs
})

submission.to_csv('/Volumes/workspace/climate_challenge/raw_files/submission_optuna_tuned.csv', index=False)
print(submission.head())
print("Shape:", submission.shape)

In [0]:
import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score

with mlflow.start_run(run_name="xgboost_optuna_champion"):
    best_params = study.best_params
    best_params['scale_pos_weight'] = scale_pos_weight
    best_params['eval_metric'] = 'logloss'
    best_params['random_state'] = 42

    champion = XGBClassifier(**best_params)
    champion.fit(X_train, y_train)

    preds = champion.predict(X_val)
    probs = champion.predict_proba(X_val)[:, 1]

    f1 = f1_score(y_val, preds)
    auc = roc_auc_score(y_val, probs)

    mlflow.log_params(best_params)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", auc)
    mlflow.xgboost.log_model(champion, "model", input_example=X_train[:5])

    print(f"F1: {f1:.4f}")
    print(f"ROC-AUC: {auc:.4f}")

In [0]:
X_test = gold_test_pd[feature_cols]

test_preds = champion.predict(X_test)
test_probs = champion.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'ID': gold_test_pd['ID'],
    'TargetF1': test_preds,
    'TargetRAUC': test_probs
})

submission.to_csv('/Volumes/workspace/climate_challenge/raw_files/submission_optuna_champion.csv', index=False)
print(submission.head())
print("Shape:", submission.shape)

In [0]:
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import numpy as np

# use the FULL training set for CV (not the train/val split from before)
X_full = gold_train_pd[feature_cols]
y_full = gold_train_pd['is_climate_sensitive']

scale_pos_weight_full = (y_full == 0).sum() / (y_full == 1).sum()

def objective_cv(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'scale_pos_weight': scale_pos_weight_full,
        'eval_metric': 'logloss',
        'random_state': 42
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    for train_idx, val_idx in skf.split(X_full, y_full):
        X_tr, X_va = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_tr, y_va = y_full.iloc[train_idx], y_full.iloc[val_idx]

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr)

        preds = model.predict(X_va)
        probs = model.predict_proba(X_va)[:, 1]

        f1 = f1_score(y_va, preds)
        auc = roc_auc_score(y_va, probs)
        fold_scores.append(0.6 * f1 + 0.4 * auc)

    return np.mean(fold_scores)

study_cv = optuna.create_study(direction='maximize')
study_cv.optimize(objective_cv, n_trials=30)

print("Best CV params:", study_cv.best_params)
print("Best mean CV score:", study_cv.best_value)

In [0]:
import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

best_cv_params = study_cv.best_params
best_cv_params['scale_pos_weight'] = scale_pos_weight_full
best_cv_params['eval_metric'] = 'logloss'
best_cv_params['random_state'] = 42

# quick sanity check on a fresh held-out split before finalizing
X_train2, X_val2, y_train2, y_val2 = train_test_split(
    X_full, y_full, test_size=0.2, random_state=7, stratify=y_full
)

with mlflow.start_run(run_name="xgboost_cv_champion"):
    champion = XGBClassifier(**best_cv_params)
    champion.fit(X_train2, y_train2)

    preds = champion.predict(X_val2)
    probs = champion.predict_proba(X_val2)[:, 1]

    f1 = f1_score(y_val2, preds)
    auc = roc_auc_score(y_val2, probs)

    mlflow.log_params(best_cv_params)
    mlflow.log_metric("cv_mean_score", study_cv.best_value)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", auc)
    mlflow.xgboost.log_model(champion, "model", input_example=X_train2[:5])

    print(f"F1: {f1:.4f}")
    print(f"ROC-AUC: {auc:.4f}")

In [0]:
champion.fit(X_full, y_full)

X_test = gold_test_pd[feature_cols]
test_preds = champion.predict(X_test)
test_probs = champion.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'ID': gold_test_pd['ID'],
    'TargetF1': test_preds,
    'TargetRAUC': test_probs
})

submission.to_csv('/Volumes/workspace/climate_challenge/raw_files/submission_cv_champion.csv', index=False)
print(submission.head())
print("Shape:", submission.shape)

In [0]:
# Test finished